In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# 1. 넘겨준 npy 파일 로드
X_raw = np.load('X32.npy')
y = np.load('y32.npy')

print(f"전체 데이터 형태: X={X_raw.shape}, y={y.shape}")

# 2. Train / Test 분할 (8:2 비율, 클래스 비율 유지)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.preprocessing import RobustScaler

# 네트워크 데이터의 극단적인 이상치(Outlier)에 강한 RobustScaler 사용
scaler = RobustScaler()

# X_train에만 fit(기준 설정)과 transform(변환)을 동시에 적용
X_train_scaled = scaler.fit_transform(X_train)

# X_test는 절대 fit하지 않고, Train의 기준에 맞춰 transform(변환)만 적용!
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# 1. 랜덤 포레스트 모델 정의
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 2. 모델 학습
print("\n[Random Forest] 학습 시작...")
rf_model.fit(X_train_scaled, y_train)

# 3. 테스트 데이터 예측
y_pred = rf_model.predict(X_test_scaled)

# 4. 평가지표 계산
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

# 5. 혼동 행렬 추출 및 오탐률(FPR) 계산 (0: 정상, 1: 공격 가정)
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn) 

# 6. 결과 저장
rf_performance = {
    "Model": "Random Forest",
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1-Score": f1,
    "False Positive Rate (FPR)": fpr
}

# 7. 결과 출력
print("Random Forest 학습 및 평가 완료!")
print(f"- Accuracy : {acc:.4f}")
print(f"- Precision: {prec:.4f}")
print(f"- Recall   : {rec:.4f}")
print(f"- F1-Score : {f1:.4f}")
print(f"- 오탐률(FPR): {fpr*100:.4f}%")